# Abdul Model Jupyter Runner

Dieses Notebook initialisiert den aktuellen Branch-Stand robust, laedt einen frischen 2%-Testdatensatz fuer Abdul und startet danach den Wrapper `run_abdul_model.py`, ohne Abduls Code unter `model-training/` zu veraendern.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo_name = 'Hackathon-TUM'
branch_name = 'feature/jupyter-integration'

cwd = Path.cwd().resolve()
while cwd.name == repo_name:
    cwd = cwd.parent
os.chdir(cwd)

repo_dir = Path.cwd() / repo_name
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Say43/Hackathon-TUM.git', repo_name], check=True)

subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', branch_name], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', branch_name], check=True)

os.chdir(repo_dir)
repo_root = Path.cwd().resolve()
print('Working directory:', repo_root)
print('Branch:')
subprocess.run(['git', 'branch', '--show-current'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_root / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_root / 'model-training' / 'requirements.txt')], check=True)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

test_data_root = repo_root / 'model-training' / 'data' / 'abdul-testrun'
test_run_root = repo_root / 'model-training' / 'runs' / 'abdul-testrun'

if test_data_root.exists():
    shutil.rmtree(test_data_root)
if test_run_root.exists():
    shutil.rmtree(test_run_root)

subprocess.run([sys.executable, str(repo_root / 'download_abdul_testrun_data.py')], check=True)

In [ ]:
data_dir = (repo_root / 'model-training' / 'data' / 'abdul-testrun' / 'makeathon-challenge').resolve()
run_dir = (repo_root / 'model-training' / 'runs' / 'abdul-testrun').resolve()

print('Repo root:', repo_root)
print('Data dir:', data_dir)
print('Exists:', data_dir.exists())
print('Run dir:', run_dir)

label_root = data_dir / 'labels' / 'train'
print('Label tif count:', sum(1 for _ in label_root.rglob('*.tif')))

if not data_dir.exists():
    raise FileNotFoundError(f'Dataset not found: {data_dir}')

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        str(repo_root / 'run_abdul_model.py'),
        'all',
        '--data-dir',
        str(data_dir),
        '--run-dir',
        str(run_dir),
    ],
    check=True,
)

In [ ]:
submission = run_dir / 'submission.geojson'
pred_dir = run_dir / 'predictions'
filtered_pred_dir = run_dir / 'filtered_predictions'

print('Submission exists:', submission.exists())
if submission.exists():
    print('Submission:', submission)

print('Prediction files:')
for path in sorted(pred_dir.glob('pred_*.tif'))[:20]:
    print('-', path)

In [ ]:
import json
import geopandas as gpd

with submission.open('r', encoding='utf-8') as f:
    submission_data = json.load(f)

gdf = gpd.GeoDataFrame.from_features(submission_data['features'], crs='EPSG:4326')
print('Anzahl Features:', len(gdf))
display(gdf.head())

In [ ]:
import rasterio
import numpy as np

pred_path = sorted(pred_dir.glob('pred_*.tif'))[0]
with rasterio.open(pred_path) as src:
    arr = src.read(1)

print('Prediction preview:', pred_path.name)
print('shape:', arr.shape)
print('dtype:', arr.dtype)
print('min:', arr.min())
print('max:', arr.max())
print('unique values:', np.unique(arr)[:20])
print('positive pixels:', int((arr > 0).sum()))

In [ ]:
import subprocess
import sys

filtered_pred_dir.mkdir(parents=True, exist_ok=True)
filtered_pred_path = filtered_pred_dir / pred_path.name

subprocess.run(
    [
        sys.executable,
        str(repo_root / 'apply_prediction_filter.py'),
        '--input',
        str(pred_path),
        '--output',
        str(filtered_pred_path),
        '--opening-size',
        '2',
        '--closing-size',
        '2',
        '--min-component-size',
        '64',
    ],
    check=True,
)

with rasterio.open(filtered_pred_path) as src:
    filtered_arr = src.read(1)

print('Filtered preview:', filtered_pred_path.name)
print('shape:', filtered_arr.shape)
print('dtype:', filtered_arr.dtype)
print('min:', filtered_arr.min())
print('max:', filtered_arr.max())
print('unique values:', np.unique(filtered_arr)[:20])
print('positive pixels before:', int((arr > 0).sum()))
print('positive pixels after:', int((filtered_arr > 0).sum()))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(arr, cmap='gray', vmin=0, vmax=max(1, arr.max()))
axes[0].set_title(f'Original: {pred_path.name}')
axes[0].axis('off')

axes[1].imshow(filtered_arr, cmap='gray', vmin=0, vmax=max(1, filtered_arr.max()))
axes[1].set_title(f'Filtered: {filtered_pred_path.name}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Frontend API Bootstrap

Diese Zellen installieren nur die minimal noetigen Backend-Abhaengigkeiten fuer das Dashboard auf der aktuellen Python-Umgebung und starten dann den bestehenden FastAPI-Adapter gegen den aktuellen Abdul-Run.

In [ ]:
import subprocess
import sys

req_path = repo_root / 'backend_runtime_requirements.txt'
print('Installing backend runtime deps from:', req_path)
print('Exists:', req_path.exists())

if not req_path.exists():
    raise FileNotFoundError(f'Missing requirements file: {req_path}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_path)], check=True)

In [ ]:
import os
import subprocess
import sys
import time
from urllib.request import urlopen
from urllib.error import URLError

backend_env = os.environ.copy()
backend_env['PRED_DIR'] = str(pred_dir)
backend_env['MAKEATHON_DATA_DIR'] = str(data_dir)

print('Starting backend at http://127.0.0.1:8000')
print('PRED_DIR =', backend_env['PRED_DIR'])
print('MAKEATHON_DATA_DIR =', backend_env['MAKEATHON_DATA_DIR'])

backend_process = subprocess.Popen(
    [sys.executable, str(repo_root / 'run_frontend_backend.py'), '--pred-dir', str(pred_dir), '--data-dir', str(data_dir), '--reload'],
    env=backend_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print('Backend PID:', backend_process.pid)

health_url = 'http://127.0.0.1:8000/api/health'
started = False
for _ in range(30):
    if backend_process.poll() is not None:
        output = backend_process.stdout.read() if backend_process.stdout else ''
        raise RuntimeError(f'Backend exited early with code {backend_process.returncode}\n\n{output}')
    try:
        with urlopen(health_url, timeout=1) as r:
            print('Backend health:', r.read().decode('utf-8'))
            started = True
            break
    except URLError:
        time.sleep(1)

if not started:
    output = ''
    if backend_process.stdout:
        output = backend_process.stdout.read()
    raise RuntimeError(f'Backend did not become ready in time.\n\n{output}')

## Frontend Dev Server Bootstrap

In manchen Jupyter-Web-Umgebungen ist Node.js / npm nicht installiert. Dann kann das Frontend hier nicht lokal gestartet werden, auch wenn Backend und Daten korrekt laufen.

In [ ]:
import shutil

npm_path = shutil.which('npm')
print('npm path:', npm_path)

if npm_path is None:
    raise RuntimeError(
        'npm is not available in this Jupyter environment. The backend is running, but the Vite frontend cannot be started here. '\
        'Start the frontend locally in VS Code with: cd frontend && npm install && npm run dev'
    )

In [ ]:
import subprocess

subprocess.run(['npm', 'install'], cwd=str(repo_root / 'frontend'), check=True)

In [ ]:
import subprocess
import time
from urllib.request import urlopen
from urllib.error import URLError

print('Starting frontend at http://127.0.0.1:5173')

frontend_process = subprocess.Popen(
    [sys.executable, str(repo_root / 'run_frontend_dev.py'), '--host', '127.0.0.1', '--port', '5173'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print('Frontend PID:', frontend_process.pid)

frontend_url = 'http://127.0.0.1:5173'
frontend_started = False
for _ in range(30):
    if frontend_process.poll() is not None:
        output = frontend_process.stdout.read() if frontend_process.stdout else ''
        raise RuntimeError(f'Frontend exited early with code {frontend_process.returncode}\n\n{output}')
    try:
        with urlopen(frontend_url, timeout=1) as r:
            print('Frontend is reachable:', frontend_url)
            frontend_started = True
            break
    except URLError:
        time.sleep(1)

if not frontend_started:
    output = ''
    if frontend_process.stdout:
        output = frontend_process.stdout.read()
    raise RuntimeError(f'Frontend did not become ready in time.\n\n{output}')